In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data extraction
import kagglehub

import os
import glob


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Timing
import time

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

from sklearn.metrics import (
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

# Dimensionality Reduction
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# Imbalanced Learning
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import ClusterCentroids
from imblearn.combine import SMOTETomek

# Feature Extraction
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
import shap

# Data manipulation
from sklearn.model_selection import StratifiedShuffleSplit

# Misc
import warnings
warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid")

In [ ]:
# Download latest version if it doesn't exist yet
dataset_key = "mlg-ulb/creditcardfraud"
csv_name = "creditcardfraud.csv"

if not os.path.exists(csv_name):
    path = kagglehub.dataset_download(dataset_key)
else:
    path = csv_name

csv_files = glob.glob(path + "/**/*.csv", recursive=True)
print(csv_files)
df = pd.read_csv(csv_files[0])

print("Dataset Shape:", df.shape)

df.head()

In [ ]:
print(df.info())
print(df.describe())

# Check for missing values
print("Missing Values:\n", df.isnull().sum()) 

In [ ]:
class_counts = df["Class"].value_counts()

print(class_counts)

plt.figure(figsize=(6,4))
sns.countplot(x="Class", data=df)
plt.title("Fraud vs Non-Fraud Transactions")
plt.show()

fraud_percentage = (
    df["Class"].sum() / len(df)
) * 100

print(f"Fraud Percentage: {fraud_percentage:.4f}%")


**Observations:**

1. Dataset contains **284,807 transactions**.
2. Fraud transactions represent only **0.1727%** of the dataset.



## Model Factory Function

Creates all classifiers used in the project for fraud detection under strong class imbalance.

We use multiple models to compare how different learning approaches behave under imbalanced data, where false negatives (missed fraud) are more costly than false positives.

To address this, we use **class weighting**, which increases the importance of fraud samples during training by scaling their contribution to the loss. This shifts models toward detecting fraud more aggressively.

---

### Models included

#### K-Nearest Neighbors (KNN)
- Baseline distance-based model
- No class weighting support
- Imbalance handled indirectly via scaling and threshold tuning

#### Logistic Regression
- Linear model with probability output
- Supports `class_weight`
- Trained with weighted loss to prioritize fraud detection

#### Random Forest
- Tree ensemble model
- Supports `class_weight`
- Uses weighted impurity in split decisions

#### AdaBoost
- Boosting ensemble model
- Uses `sample_weight` instead of `class_weight`
- Fraud samples are given higher weight during boosting to force focus on minority class

In [ ]:
def create_models(random_state=42, class_weight=None):
    """
    Creates and returns all classifiers used in the project.
    class_weight: dict like {0:1, 1:10} or "balanced"
    """

    models = {
        "KNN": KNeighborsClassifier(
            n_neighbors=5
        ),

        "Logistic Regression": LogisticRegression(
            max_iter=1000,
            random_state=random_state,
            class_weight=class_weight
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1,
            class_weight=class_weight
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=100,
            random_state=random_state
        )
    }

    return models

## Data Balancing Factory Function (Q1)

This function creates and returns all balancing techniques used to handle class imbalance in the dataset.

### Methods included:
- None (no balancing)
- Undersampling (RandomUnderSampler)
- SMOTE (Synthetic Minority Over-sampling Technique)
- SMOTE-Tomek (hybrid approach combining SMOTE and Tomek links)

In [ ]:
def create_balancing_methods(random_state=42):

    return {
        "None": None,

        "Undersampling": RandomUnderSampler(
            random_state=random_state
        ),

        "SMOTE": SMOTE(
            random_state=random_state
        ),

        "SMOTE_Tomek": SMOTETomek(
            random_state=random_state
        ),
        "ClusterCentroids": ClusterCentroids(
            random_state=random_state
        )
    }

## Threshold Optimization for Recall Bias

In fraud detection, it is preferable to incorrectly flag a transaction as fraud
rather than miss an actual fraudulent transaction.

Therefore, we adjust the decision threshold of the model to prioritize recall.

Recall is defined as:

Recall = TP / (TP + FN)

By maximizing recall, we minimize false negatives (missed fraud cases),
which is critical in real-world fraud detection systems.

We therefore select the smallest threshold that satisfies a target recall,
making the classifier more sensitive to fraud predictions.

In [ ]:
def find_threshold_for_target_recall(y_true, y_proba, target_recall=0.95):
    """
    Returns the lowest threshold that achieves target recall.
    If not achievable, returns the smallest possible threshold.
    """

    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)

    # align sizes (recall has +1 element vs thresholds)
    recall = recall[:-1]

    valid_idx = np.where(recall >= target_recall)[0]

    if len(valid_idx) > 0:
        # pick highest threshold that still satisfies recall
        best_idx = valid_idx[-1]
        return thresholds[best_idx]
    
    # fallback: most aggressive (catch everything)
    return thresholds[np.argmin(recall)]

## Apply Balancing To Data

Function to apply the balance methods to the data before running models

In [ ]:
def apply_balancing(X_train, y_train, method):
    """
    Applies balancing ONLY on training data.
    """

    if method is None:
        return X_train, y_train

    X_res, y_res = method.fit_resample(X_train, y_train)

    return X_res, y_res

### Feature Importance Extraction (Q2+3)

This function extracts feature importance from trained models.

- Random Forest / AdaBoost → uses built-in `feature_importances_`
- Logistic Regression → uses absolute values of coefficients

Output:
A sorted table of features ranked by importance, showing how strongly each feature influences predictions.

In [ ]:
def extract_feature_importance(model, feature_names, model_name="model"):
    """
    Extracts feature importance from supported models:
    - RandomForest
    - AdaBoost
    - Logistic Regression
    """

    importance = None

    # Tree-based models
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_

    # Linear models
    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_).ravel()

    else:
        raise ValueError(f"Model {model_name} does not support feature importance extraction")

    df = pd.DataFrame({
        "feature": feature_names,
        "importance": importance
    })

    df = df.sort_values("importance", ascending=False).reset_index(drop=True)

    return df

### SHAP Feature Analysis

SHAP helps explaining model predictions by assigning each feature a contribution value.

- `compute_shap_values`: calculates SHAP values for each prediction
- `shap_global_importance`: aggregates SHAP values into global feature importance

Output:
- Local explanations (per prediction)
- Global feature ranking (mean absolute SHAP value)


In [ ]:
def compute_shap_values(model, X_train, X_test):
    """
    Computes SHAP values depending on model type.
    """

    explainer = shap.Explainer(model, X_train)
    shap_values = explainer(X_test)

    return shap_values


def shap_global_importance(shap_values, feature_names):
    """
    Converts SHAP values into global feature importance.
    """

    importance = np.abs(shap_values.values).mean(axis=0)

    df = pd.DataFrame({
        "feature": feature_names,
        "shap_importance": importance
    }).sort_values("shap_importance", ascending=False)

    return df

### Correlation Analysis

This function measures relationships between features and the target using:

- Pearson: linear correlation
- Spearman: rank-based (monotonic) correlation
- Mutual Information: general nonlinear dependency

Output:
A ranked list of features showing how strongly each relates to fraud detection.

In [ ]:
def correlation_analysis(X, y, method="pearson"):
    """
    Computes feature correlation with target or between features.
    """

    df = pd.DataFrame(X)

    if method == "pearson":
        return df.corrwith(pd.Series(y)).sort_values(ascending=False)

    elif method == "spearman":
        return df.corrwith(pd.Series(y), method="spearman").sort_values(ascending=False)

    elif method == "mutual_info":
        mi = mutual_info_classif(X, y, random_state=42)
        return pd.Series(mi, index=df.columns).sort_values(ascending=False)

    else:
        raise ValueError("method must be: pearson | spearman | mutual_info")

### Full Feature Analysis Pipeline

This function combines all analysis methods into one pipeline:

- Model-based importance (RF / AdaBoost / Logistic Regression)
- SHAP explainability
- Pearson, Spearman, and Mutual Information correlation

Output:
A complete feature analysis report used for:
- feature ranking
- interpretability
- dataset understanding

In [ ]:
def full_feature_report(model, X_train, X_test, y_train, y_test, feature_names):
    """
    Combines:
    - Model importance
    - SHAP importance
    - Correlations
    """

    report = {}

    # Model importance
    report["model_importance"] = extract_feature_importance(
        model, feature_names
    )

    # SHAP
    shap_values = compute_shap_values(model, X_train, X_test)
    report["shap_importance"] = shap_global_importance(shap_values, feature_names)

    # Correlations
    report["pearson"] = correlation_analysis(X_train, y_train, "pearson")
    report["spearman"] = correlation_analysis(X_train, y_train, "spearman")
    report["mutual_info"] = correlation_analysis(X_train, y_train, "mutual_info")

    return report

## Helper Methods (Q1-3)

This section contains utility functions used throughout the experiment pipeline.
They handle:
- preprocessing
- balancing
- evaluation
- feature analysis
- experiment management 


Creates all baseline models used in the experiment with optional class weighting

In [ ]:
def build_model(model_name, class_weight, random_state=42):

    models = create_models(
        random_state=random_state,
        class_weight=class_weight
    )

    return clone(models[model_name])

Evaluates model performance using classification metrics and threshold strategies.

In [ ]:
def evaluate_model(y_true, y_pred, y_proba=None):
    """
    Computes a comprehensive set of evaluation metrics for classification.
    Designed for imbalanced fraud detection tasks.
    """

    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

    # Threshold-independent metrics (if probabilities available)
    if y_proba is not None:
        results["roc_auc"] = roc_auc_score(y_true, y_proba)
        results["pr_auc"] = average_precision_score(y_true, y_proba)

    # Confusion matrix breakdown (useful for fraud analysis)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    results.update({
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

    return results

Extracts feature importance from Correlations

In [ ]:
def compute_dataset_feature_stats(X, y, feature_names):

    return {
        "pearson": correlation_analysis(X, y, "pearson"),
        "spearman": correlation_analysis(X, y, "spearman"),
        "mutual_info": pd.Series(
            mutual_info_classif(X, y, random_state=42),
            index=feature_names
        ).sort_values(ascending=False)
    }

Extracts feature importance from Models and Shap

In [ ]:
def compute_model_feature_stats(model, X_train, X_val, feature_names):

    report = {}

    # Model-specific importance (OPTION C: keep as-is)
    report["importance"] = extract_feature_importance(model, feature_names)

    # SHAP on unseen validation data (FIXED leakage)
    try:
        shap_values = compute_shap_values(model, X_train, X_val)
        report["shap"] = shap_global_importance(shap_values, feature_names)
    except:
        report["shap"] = None

    return report

Runs model evaluation across multiple threshold strategies and returns both predictions and metrics

In [ ]:
def evaluate_model_full(model, X_test, y_test, threshold_modes, target_recall=0.95):

    outputs = []

    proba = (
        model.predict_proba(X_test)[:, 1]
        if hasattr(model, "predict_proba")
        else None
    )

    for mode in threshold_modes:

        if mode == "recall" and proba is not None:
            threshold = find_threshold_for_target_recall(y_test, proba, target_recall)
            preds = (proba >= threshold).astype(int)
        else:
            preds = model.predict(X_test)

        metrics = evaluate_model(y_test, preds, proba)
        outputs.append((mode, metrics))

    return outputs

Reduces large feature importance outputs into compact top-k dictionaries for memory-efficient logging

In [ ]:
def compress_series(s, k=10):
    """
    Keeps only top-k values from a ranked series.
    Ensures memory-safe logging.
    """
    return s.head(k).to_dict()

Defines models, preprocessing methods, and evaluation settings for the experiment.

In [ ]:
def create_experiment_config(random_state=42):

    return {
        "models": create_models(random_state=random_state, class_weight=None),

        "balancing": create_balancing_methods(random_state=random_state),

        "class_weight_modes": {
            "None": None,
            "Balanced": "balanced"
        },

        "threshold_modes": ["default", "recall"],

        "checkpoint_path": "out/checkpoint.csv"
    }

## Run Q1–Q3 Experiment

Runs full model evaluation over:

models
balancing methods
class weights
threshold strategies

Also collects feature importance and SHAP summaries (for Q3)

In [ ]:
def run_q1_3(
    X,
    y,
    feature_names,
    config,
    top_k=10,
    run_shap=True,
    max_experiments=None
):

    results = []
    feature_logs = []

    models = config["models"]
    balancers = config["balancing"]
    class_weights = config["class_weight_modes"]
    threshold_modes = config["threshold_modes"]

    checkpoint_path = config.get("checkpoint_path", "checkpoint.csv")

    experiment_count = 0

    # =====================================================
    # FIXED GLOBAL SPLIT 
    # =====================================================
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # =====================================================
    # DATASET-LEVEL STATS (computed ONCE)
    # =====================================================
    dataset_stats_full = compute_dataset_feature_stats(
        X_train_full,
        y_train_full,
        feature_names
    )

    dataset_stats = {
        "pearson_top": compress_series(dataset_stats_full["pearson"], top_k),
        "spearman_top": compress_series(dataset_stats_full["spearman"], top_k),
        "mi_top": compress_series(dataset_stats_full["mutual_info"], top_k),
    }

    # =====================================================
    # LOOP: MODELS
    # =====================================================
    for m_i, (model_name, _) in enumerate(models.items(), 1):

        print(f"\n==== MODEL {m_i}/{len(models)}: {model_name} ====")

        # =================================================
        # LOOP: BALANCING
        # =================================================
        for b_i, (bal_name, balancer) in enumerate(balancers.items(), 1):

            print(f"  ---- BALANCING {b_i}/{len(balancers)}: {bal_name} ----")

            # apply balancing ONCE per configuration state
            X_train_bal, y_train_bal = apply_balancing(
                X_train_full,
                y_train_full,
                balancer
            )

            # =================================================
            # LOOP: CLASS WEIGHTS
            # =================================================
            for cw in class_weights:

                print(f"    CLASS_WEIGHT: {cw}")

                # skip invalid redundant combinations (clean design rule)
                if bal_name in {"SMOTE", "SMOTE_Tomek", "ClusterCentroids"} and cw == "balanced":
                    continue

                model = build_model(model_name, cw)
                model.fit(X_train_bal, y_train_bal)

                # =================================================
                # EVALUATION (Q1)
                # =================================================
                outputs = evaluate_model_full(
                    model,
                    X_test,
                    y_test,
                    threshold_modes
                )

                # =================================================
                # SHAP 
                # =================================================
                model_stats = {"importance": None, "shap": None}

                if run_shap:

                    # deterministic SHAP split 
                    X_shap_tr, X_shap_val, y_shap_tr, y_shap_val = train_test_split(
                        X_train_bal,
                        y_train_bal,
                        test_size=0.2,
                        random_state=42,
                        stratify=y_train_bal
                    )

                    model_stats_raw = compute_model_feature_stats(
                        model,
                        X_shap_tr,
                        X_shap_val,
                        feature_names
                    )

                    model_stats = {
                        "importance": model_stats_raw["importance"],
                        "shap": model_stats_raw["shap"]
                    }

                # =================================================
                # COMPRESS FEATURE INFO (MEMORY SAFE)
                # =================================================
                model_stats_compressed = {
                    "importance_top": compress_series(
                        model_stats["importance"],
                        top_k
                    ) if model_stats["importance"] is not None else None,

                    "shap_top": compress_series(
                        model_stats["shap"],
                        top_k
                    ) if model_stats["shap"] is not None else None
                }

                # =================================================
                # STORE FEATURE LOGS (NO REDUNDANCY)
                # =================================================
                feature_logs.append({
                    "model": model_name,
                    "balancing": bal_name,
                    "class_weight": cw,
                    "dataset_stats": dataset_stats,
                    "model_stats": model_stats_compressed
                })

                # =================================================
                # STORE RESULTS (Q1)
                # =================================================
                for mode, metrics in outputs:
                    row = {
                        "model": model_name,
                        "balancing": bal_name,
                        "class_weight": cw,
                        "threshold_mode": mode,
                        **metrics
                    }

                    results.append(row)

                    # =================================================
                    # CHECKPOINT (CRASH SAFE, NO REDUNDANCY)
                    # =================================================
                    pd.DataFrame([row]).to_csv(
                        checkpoint_path,
                        mode="a",
                        header=not os.path.exists(checkpoint_path),
                        index=False
                    )

                # =================================================
                # EXPERIMENT LIMIT (SAFETY)
                # =================================================
                experiment_count += 1
                if max_experiments is not None and experiment_count >= max_experiments:
                    print("Stopping early (max_experiments reached)")
                    return results, feature_logs

    return results, feature_logs

### Feature Selection Methods

This function defines feature selection strategies used before model training.

We compare:
- No feature selection (baseline)
- Mutual Information (captures nonlinear relationships)
- ANOVA F-test (linear relationship baseline)

All methods are applied only on the training set to avoid data leakage.

Purpose:
- Reduce noise
- Test whether dimensionality reduction improves fraud detection
- Compare linear vs nonlinear feature relevance

In [ ]:
def create_selectors():
    """
    Feature selection methods used in the project.
    """

    return {
        "None": None,

        "MutualInfo": SelectKBest(
            score_func=mutual_info_classif,
            k=10
        ),

        "F_Classif": SelectKBest(
            score_func=f_classif,
            k=10
        )
    }

## Apply selector  
Helper func to apply the selector on the data

In [ ]:
def apply_selector(selector, X_train, y_train, X_test):
    if selector is None:
        return X_train, X_test
    X_train = selector.fit_transform(X_train, y_train)
    X_test = selector.transform(X_test)
    return X_train, X_test

## Training Set Size Strategies (Q4)

This function defines how we vary the amount of training data to study learning behavior and performance saturation.

We use **stratified random sampling** to ensure each subset preserves the original fraud/normal ratio.

We test progressively larger training sizes:
- 1% , 2% , 5% , 10% , 20% , 40%, 60% , 80%, 90%

The test set remains fixed for all experiments to ensure fair comparison.

Goal:
Determine at what point adding more training data yields minimal performance improvement.

In [ ]:
TRAIN_SIZES = [0.01, 0.02, 0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 0.90]

## Training Set Modes

Defines how training subsets are generated.

- `"Default"` uses the full dataset (no sampling)
- Percentage modes (1%–90%) use `StratifiedShuffleSplit` to create stratified subsets

Each mode returns a splitter configured with:
- fixed `train_size`
- stratified sampling
- configurable `n_splits` and `random_state`  
  - `n_splits`: number of repeated samples per training size  
  - `random_state`: controls randomness for reproducible splits

Used as the configuration step before generating actual training sets.

In [ ]:
def create_training_set_modes(random_state=42, n_splits=1):
    return {
        "Default": None,
        **{
            f"{int(size*100)}%": StratifiedShuffleSplit(
                n_splits=n_splits,
                train_size=size,
                random_state=random_state
            )
            for size in TRAIN_SIZES
        }
    }

## Applying Modes On Sets

Applies the predefined `modes` to the dataset to generate actual training subsets.

Each mode defines how the data is sampled (full data or stratified percentage splits), and this function materializes those configurations into `(X_train, y_train)` pairs.


In [ ]:
def build_training_sets(X, y, modes):
    training_sets = {}

    for mode_name, splitter in modes.items():

        # Default = full training data
        if splitter is None:
            training_sets[mode_name] = [(X, y)]
            continue

        sets = []

        # StratifiedShuffleSplit can yield multiple splits
        for train_idx, _ in splitter.split(X, y):
            X_train = X[train_idx]
            y_train = y[train_idx]
            sets.append((X_train, y_train))

        training_sets[mode_name] = sets

    return training_sets

## Build Training Splits (Wrapper)

Wrapper function that builds the full set of training datasets in one step.

- creates training modes (sampling strategies and sizes)
- applies them to `(X, y)`
- returns all generated training sets

In [ ]:
def create_training_pipeline(X, y, random_state=42, n_splits=1):
    modes = create_training_set_modes(random_state, n_splits)
    return build_training_sets(X, y, modes)

## Temporal Data Splitting (Q5)

This function defines different temporal evaluation strategies for fraud detection.

It supports multiple ways of splitting data over time:

* Forward training: train on past data and test on future data 
* Fixed test + segment training: same test set, different time-based training segments
* Rolling window: sliding window over time to evaluate local performance drift

### Used to evaluate:

* Concept drift
* Model stability over time
* Performance changes across time periods

In [ ]:
def create_time_splits(X, y, time_col, mode="forward", n_splits=5, window_size=0.2):
    """
    Returns list of (train_idx, test_idx) pairs depending on temporal strategy.
    """

    # sort by time
    order = np.argsort(X[time_col])
    X_sorted = X[order]
    y_sorted = y[order]

    n = len(X_sorted)

    splits = []

    # ----------------------------
    # 1. FORWARD TRAINING
    # ----------------------------
    if mode == "forward":
        step = n // (n_splits + 1)

        for i in range(1, n_splits + 1):
            train_end = i * step
            test_end = (i + 1) * step if i < n_splits else n

            train_idx = np.arange(0, train_end)
            test_idx = np.arange(train_end, test_end)

            splits.append((train_idx, test_idx))

    # ----------------------------
    # 2. FIXED TEST + SEGMENTS
    # ----------------------------
    elif mode == "fixed_test_segments":
        test_size = int(0.2 * n)
        test_idx = np.arange(n - test_size, n)

        train_pool = np.arange(0, n - test_size)
        segment_size = len(train_pool) // n_splits

        for i in range(n_splits):
            start = i * segment_size
            end = (i + 1) * segment_size

            train_idx = train_pool[start:end]
            splits.append((train_idx, test_idx))

    # ----------------------------
    # 3. ROLLING WINDOW
    # ----------------------------
    elif mode == "rolling":
        window = int(window_size * n)

        step = window // 2

        for start in range(0, n - window - step, step):
            train_idx = np.arange(start, start + window)
            test_idx = np.arange(start + window, start + window + step)

            splits.append((train_idx, test_idx))

    return splits

## Feature Importance Drift Over Time 

This function analyzes how feature importance changes over time.

It aggregates feature importance values from different time segments and computes:

* Mean importance: overall feature influence
* Standard deviation: how unstable a feature is across time
* Drift: how much feature importance changes over time

### Used to identify:

* Consistently important features
* Time-dependent features
* Evolving fraud patterns


In [ ]:
def analyze_feature_drift(feature_importances_over_time, feature_names):
    """
    Converts importance over time into a table.
    """

    df = pd.DataFrame(feature_importances_over_time, columns=feature_names)

    return {
        "mean_importance": df.mean().sort_values(ascending=False),
        "std_importance": df.std().sort_values(ascending=False),
        "drift": df.diff().abs().mean().sort_values(ascending=False)
    }

## Model Evaluation Metrics
This function computes a complete set of evaluation metrics to assess classifier performance, with emphasis on fraud detection where class imbalance is severe.

It includes both threshold-dependent and threshold-independent metrics, as well as detailed error breakdown.

### Accuracy, Precision, Recall, F1-score

Standard classification metrics computed using predicted labels:

* Accuracy: overall correctness (often misleading in imbalanced data)
* Precision: how many predicted fraud cases are actually fraud
* Recall: how many actual fraud cases are detected (critical for this project)
* F1-score: balance between precision and recall

### ROC-AUC (Receiver Operating Characteristic Area Under Curve)

Measures the model’s ability to rank positive cases (fraud) higher than negative ones across all thresholds.

In this project, it is useful for:

* evaluating overall separability between fraud and non-fraud
* comparing models independently of threshold choice

### PR-AUC (Precision-Recall Area Under Curve)

Measures precision-recall trade-off across all thresholds.

In this project, it is especially important because:

* the dataset is highly imbalanced
* ROC-AUC can appear overly optimistic
* PR-AUC better reflects performance on the minority (fraud) class

### Confusion Matrix Breakdown

Decomposes predictions into:

* True Negatives (correct normal transactions)
* False Positives (normal flagged as fraud)
* False Negatives (fraud missed)
* True Positives (fraud correctly detected)

This is essential in our case because:

* false negatives represent missed fraud cases, which are the most critical errors
* it allows direct interpretation of model failure types



## Helper Functions (TODO - Ignore for now)
We'll now define helper methods to assist the running of the main experiment including:
- Time managment
- Redundancy skip (e.g SMOTE makes class_weight "None" and "Balanced" the same)
- Progress save (Writing to a file the current run's progress data)
- Final results save (Writing to a file the run's data once the run ended)

In [ ]:
def flatten_feature_importance(feature_importance):
    """
    Converts feature importance log into tabular format.
    """

    return [
        {
            "model": e.get("model"),
            "balancing": e.get("balancing"),
            "class_weight": e.get("class_weight"),
            "selector": e.get("selector"),
            "time_mode": e.get("time_mode"),
        }
        for e in feature_importance
    ]

In [ ]:
def format_elapsed(start_time):
    """
    Converts a start timestamp into human-readable elapsed time.
    """

    elapsed = time.time() - start_time

    hours = int(elapsed // 3600)
    minutes = int((elapsed % 3600) // 60)
    seconds = elapsed % 60

    return f"{hours:02}:{minutes:02}:{seconds:05.2f}"

In [ ]:
CLASS_WEIGHT_MODES = {
    "None": None,
    "Balanced": "balanced"
}

SKIP_BALANCED_CLASS_WEIGHT = {
    "SMOTE",
    "SMOTE_Tomek",
    "ClusterCentroids"
}


def should_skip_class_weight(balancing_name, class_weight_name):
    """
    Returns True if this combination is redundant and should be skipped.
    """

    return (
        class_weight_name == "Balanced"
        and balancing_name in SKIP_BALANCED_CLASS_WEIGHT
    )

In [ ]:
def update_progress(progress_path, **kwargs):
    """
    Writes current experiment state (debug / crash recovery log).
    """

    with open(progress_path, "w") as f:
        for k, v in kwargs.items():
            f.write(f"{k}: {v}\n")
            
def save_checkpoint(results, path):
    pd.DataFrame(results).to_csv(path, index=False)

In [ ]:
def save_final_results(results, feature_importance, results_path, feature_path):
    """
    Saves final experiment outputs (final aggregation step).
    """

    pd.DataFrame(results).to_csv(results_path, index=False)

    feature_df = pd.DataFrame(flatten_feature_importance(feature_importance))
    feature_df.to_csv(feature_path, index=False)

In [ ]:
def train_and_evaluate(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    selector,
    threshold_modes
):
    """
    Handles:
    - training
    - feature selection
    - prediction
    - thresholding
    - evaluation
    """

    if selector is not None:
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    outputs = []

    for mode in threshold_modes:

        if mode == "recall" and proba is not None:
            threshold = find_threshold_for_target_recall(y_test, proba)
            preds = (proba >= threshold).astype(int)
        else:
            preds = model.predict(X_test)

        metrics = evaluate_model(y_test, preds, proba)

        outputs.append((mode, metrics))

    return outputs, proba

In [ ]:
def evaluate_split(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    selector,
    threshold_modes,
    balancer,
    extra_tag=None
):
    """
    Shared training + evaluation logic for Q4 and Q5.
    """

    X_train, y_train = apply_balancing(X_train, y_train, balancer)

    evals, _ = train_and_evaluate(
        model,
        X_train, y_train,
        X_test, y_test,
        selector,
        threshold_modes
    )

    results = []

    for mode, metrics in evals:
        results.append({
            "threshold_mode": mode,
            "metrics": metrics,
            "tag": extra_tag
        })

    return results

## Experiment Configuration 

This section defines all configurable components used in the experiment pipeline

contains:
- Output files paths
- models
- balancing
- feature selection
- time strategies
- training strategies


In [ ]:
def create_experiment_config(random_state=42):

    return {

        # -----------------------
        # MODELS
        # -----------------------
        "models": create_models(
            random_state=random_state,
            class_weight=None
        ),

        # -----------------------
        # BALANCING (Q1)
        # -----------------------
        "balancing": create_balancing_methods(
            random_state=random_state
        ),

        # -----------------------
        # SELECTORS (Q2/Q3)
        # -----------------------
        "selectors": create_selectors(),

        # -----------------------
        # TRAIN SIZE (Q4)
        # -----------------------
        "training_modes": create_training_set_modes(
            random_state=random_state,
            n_splits=1
        ),

        # -----------------------
        # TIME SPLITS (Q5)
        # -----------------------
        "time_modes": ["forward", "rolling", "fixed_test_segments"],

        # -----------------------
        # THRESHOLDS
        # -----------------------
        "threshold_modes": ["default", "recall"],

        # -----------------------
        # CLASS WEIGHT OPTIONS
        # -----------------------
        "class_weight_modes": [None, "balanced"],
        
        # -----------------------
        # AMOUNT MODES
        # -----------------------
        "amount_modes": {
            "with_amount": True,
            "without_amount": False
        },
        
        # -----------------------
        # OUTPUT PATHS
        # -----------------------
        "paths": {
            "results": "out/experiment_results.csv",
            "checkpoint": "out/checkpoint.csv",
            "progress": "out/progress.txt",
            "features": "out/feature_importance.csv"
        }
    }

In [ ]:
def run_experiment(
    X,
    y,
    feature_names,
    config,
    time_col="Time",
    checkpoint_every=50
):

    results = []
    feature_logs = []

    start_time = time.time()
    step_counter = 0

    # -----------------------
    # CONFIG
    # -----------------------
    models_dict = config["models"]
    balancing_methods = config["balancing"]
    selectors = config["selectors"]
    training_modes = config["training_modes"]
    time_modes = config["time_modes"]
    threshold_modes = config["threshold_modes"]
    class_weight_modes = config["class_weight_modes"]
    amount_modes = config["amount_modes"]
    paths = config["paths"]

    total_models = len(models_dict)

    # =========================================================
    # MODEL LOOP
    # =========================================================
    for m_i, (model_name, _) in enumerate(models_dict.items(), 1):

        print(f"\n==== MODEL {m_i}/{total_models}: {model_name} ====")

        # =====================================================
        # BALANCING LOOP
        # =====================================================
        for b_i, (bal_name, balancer) in enumerate(balancing_methods.items(), 1):

            print(f"  ---- BALANCING {b_i}/{len(balancing_methods)}: {bal_name} ----")

            # =================================================
            # CLASS WEIGHT LOOP
            # =================================================
            for class_weight in class_weight_modes:

                if should_skip_class_weight(bal_name, class_weight):
                    continue

                base_model = create_models(
                    random_state=42,
                    class_weight=class_weight
                )[model_name]

                # =================================================
                # SELECTOR LOOP
                # =================================================
                for sel_name, selector_template in selectors.items():

                    print(f"    SELECTOR: {sel_name}")

                    # =================================================
                    # Q2: AMOUNT ABLATION
                    # =================================================
                    for amount_mode, use_amount in amount_modes.items():

                        if not use_amount:
                            amount_idx = list(feature_names).index("Amount")
                            X_run = np.delete(X, amount_idx, axis=1)
                            feature_names_run = [
                                f for f in feature_names if f != "Amount"
                            ]
                        else:
                            X_run = X
                            feature_names_run = feature_names

                        # =================================================
                        # Q4: TRAIN SIZE EXPERIMENT
                        # =================================================
                        print("      Running Q4")

                        for splitter in training_modes.values():

                            splits = (
                                [(np.arange(len(X_run)), None)]
                                if splitter is None
                                else splitter.split(X_run, y)
                            )

                            for train_idx, _ in splits:

                                X_train, y_train = X_run[train_idx], y[train_idx]

                                X_test = X_run
                                y_test = y

                                # fresh model per run
                                model = clone(base_model)

                                # fresh selector per run
                                selector = (
                                    None
                                    if selector_template is None
                                    else clone(selector_template)
                                )

                                # apply selector
                                X_train, X_test = apply_selector(
                                    selector,
                                    X_train,
                                    y_train,
                                    X_test
                                )

                                # balancing ONLY train
                                X_bal, y_bal = apply_balancing(
                                    X_train, y_train, balancer
                                )

                                outputs, _ = train_and_evaluate(
                                    model,
                                    X_bal,
                                    y_bal,
                                    X_test,
                                    y_test,
                                    selector,
                                    threshold_modes
                                )

                                # -------------------------
                                # FEATURE ANALYSIS (Q3)
                                # -------------------------
                                try:
                                    shap_values = compute_shap_values(
                                        model, X_bal, X_test
                                    )

                                    shap_df = shap_global_importance(
                                        shap_values,
                                        feature_names_run
                                    )

                                except:
                                    shap_df = None

                                feature_logs.append({
                                    "model": model_name,
                                    "balancing": bal_name,
                                    "class_weight": class_weight,
                                    "selector": sel_name,
                                    "amount_mode": amount_mode,
                                    "model_importance": extract_feature_importance(
                                        model, feature_names_run
                                    ),
                                    "shap_importance": shap_df,
                                    "pearson": correlation_analysis(
                                        X_bal, y_bal, "pearson"
                                    ),
                                    "spearman": correlation_analysis(
                                        X_bal, y_bal, "spearman"
                                    ),
                                    "mutual_info": correlation_analysis(
                                        X_bal, y_bal, "mutual_info"
                                    )
                                })

                                for mode, metrics in outputs:
                                    results.append({
                                        "model": model_name,
                                        "balancing": bal_name,
                                        "class_weight": class_weight,
                                        "selector": sel_name,
                                        "amount_mode": amount_mode,
                                        "split": "Q4",
                                        "threshold_mode": mode,
                                        **metrics
                                    })

                        # =================================================
                        # Q5: TIME EXPERIMENTS
                        # =================================================
                        print("      Running Q5")

                        for mode in time_modes:

                            splits = create_time_splits(
                                X_run, y,
                                time_col=time_col,
                                mode=mode
                            )

                            # fixed test only for segment mode
                            if mode == "fixed_test_segments":
                                test_cut = int(0.2 * len(X_run))
                                X_test_global = X_run[-test_cut:]
                                y_test_global = y[-test_cut:]

                            for train_idx, test_idx in splits:

                                X_train, y_train = X_run[train_idx], y[train_idx]

                                if mode == "fixed_test_segments":
                                    X_test, y_test = X_test_global, y_test_global
                                else:
                                    X_test, y_test = X_run[test_idx], y[test_idx]

                                model = clone(base_model)

                                selector = (
                                    None
                                    if selector_template is None
                                    else clone(selector_template)
                                )

                                X_train, X_test = apply_selector(
                                    selector,
                                    X_train,
                                    y_train,
                                    X_test
                                )

                                X_bal, y_bal = apply_balancing(
                                    X_train, y_train, balancer
                                )

                                outputs, _ = train_and_evaluate(
                                    model,
                                    X_bal,
                                    y_bal,
                                    X_test,
                                    y_test,
                                    selector,
                                    threshold_modes
                                )

                                # -------------------------
                                # FEATURE ANALYSIS (Q3)
                                # -------------------------
                                try:
                                    shap_values = compute_shap_values(
                                        model, X_bal, X_test
                                    )

                                    shap_df = shap_global_importance(
                                        shap_values,
                                        feature_names_run
                                    )

                                except:
                                    shap_df = None

                                feature_logs.append({
                                    "model": model_name,
                                    "balancing": bal_name,
                                    "class_weight": class_weight,
                                    "selector": sel_name,
                                    "amount_mode": amount_mode,
                                    "model_importance": extract_feature_importance(
                                        model, feature_names_run
                                    ),
                                    "shap_importance": shap_df,
                                    "pearson": correlation_analysis(
                                        X_bal, y_bal, "pearson"
                                    ),
                                    "spearman": correlation_analysis(
                                        X_bal, y_bal, "spearman"
                                    ),
                                    "mutual_info": correlation_analysis(
                                        X_bal, y_bal, "mutual_info"
                                    )
                                })

                                for mode_t, metrics in outputs:
                                    results.append({
                                        "model": model_name,
                                        "balancing": bal_name,
                                        "class_weight": class_weight,
                                        "selector": sel_name,
                                        "amount_mode": amount_mode,
                                        "split": f"Q5-{mode}",
                                        "threshold_mode": mode_t,
                                        **metrics
                                    })

                        # =================================================
                        # CHECKPOINT
                        # =================================================
                        step_counter += 1
                        if step_counter % checkpoint_every == 0:
                            save_checkpoint(results, paths["checkpoint"])

                        update_progress(
                            paths["progress"],
                            model=model_name,
                            balancing=bal_name,
                            class_weight=class_weight,
                            selector=sel_name,
                            amount_mode=amount_mode,
                            elapsed=format_elapsed(start_time)
                        )

    # =========================================================
    # FINAL SAVE
    # =========================================================
    pd.DataFrame(results).to_csv(paths["results"], index=False)
    pd.DataFrame(feature_logs).to_csv(paths["feature_importance"], index=False)

    print("\nDONE")
    print("Total runtime:", format_elapsed(start_time))

    return results, feature_logs